# 🌍 Carte céleste des alertes `sn_near_galaxy_candidate` — Fink/LSST

Ce notebook affiche la **localisation sur une projection de Mollweide** des alertes `sn_near_galaxy_candidate` récupérées depuis le portail Fink/LSST.

Pour chaque objet :
 ● position de l'**alerte** (même coordonnées ou légèrement décalées si disponibles), colorée selon le filtre dominant
- annotations optionnelles : `diaObjectId`, coordonnées α/δ

Superpositions disponibles : **plan galactique**, **écliptique**, **grille RA/Dec**.

**API :** `https://api.lsst.fink-portal.org`  
Endpoints : `/api/v1/tags` · `/api/v1/objects` · `/api/v1/sources`

In [ ]:
import requests, json

r = requests.get("https://api.lsst.fink-portal.org/api/v1/tags")
tags = r.json()
for tag, doc in tags.items():
    print(f"{tag:40s}  {doc}")

## 1. Paramètres

In [ ]:
# ─── Paramètres utilisateur ───────────────────────────────────────────────────
N_ALERTS   = 500                        # Nombre d'alertes à récupérer
STARTDATE  = None                      # "2026-01-01 00:00:00" ou None
STOPDATE   = None                      # "2026-03-01 00:00:00" ou None
BASE_URL   = "https://api.lsst.fink-portal.org"
TAG        = "extragalactic_lt20mag_candidate"# "sn_near_galaxy_candidate"

# ─── Options d'affichage ──────────────────────────────────────────────────────
SHOW_GALACTIC_PLANE = True   # Tracer le plan galactique (b = 0°)
SHOW_ECLIPTIC       = True   # Tracer l'écliptique
SHOW_GRID           = True   # Grille RA/Dec (graticules)
SHOW_RADEC_LABELS   = False   # Annoter chaque point avec α/δ
SHOW_OBJECT_ID      = False  # Annoter chaque point avec le diaObjectId (peut surcharger)
COLOR_BY            = 'filter'  # 'filter' | 'flux'  — couleur des marqueurs d'alerte

SAVE_FIGURE = True
OUTPUT_DIR  = "skymap_outputs"
# ─────────────────────────────────────────────────────────────────────────────

## 2. Imports

In [ ]:
import os
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from IPython.display import display as ipy_display

# Astropy pour les conversions de coordonnées
from astropy.coordinates import SkyCoord, get_body_barycentric
from astropy.coordinates import BarycentricTrueEcliptic
import astropy.units as u
from astropy.time import Time

warnings.filterwarnings('ignore')

if SAVE_FIGURE:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

mpl.rcParams.update({
    'font.size'        : 11,
    'axes.titlesize'   : 13,
    'axes.titleweight' : 'bold',
    'figure.titlesize' : 14,
    'figure.titleweight': 'bold',
})

FILTER_COLORS = {
    'u': '#7B2FBE',
    'g': '#0077BB',
    'r': '#33AA77',
    'i': '#DDAA33',
    'z': '#BB5500',
    'y': '#AA0000',
}

print("Imports OK ✓")

## 3. Récupération des alertes et des coordonnées

In [ ]:
def fetch_tag_alerts(tag, n, startdate=None, stopdate=None):
    payload = {"tag": tag, "n": n, "output-format": "json"}
    if startdate:
        payload["startdate"] = startdate
    if stopdate:
        payload["stopdate"] = stopdate
    resp = requests.post(f"{BASE_URL}/api/v1/tags", json=payload, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    if not data:
        raise ValueError(f"Aucune alerte pour le tag '{tag}'.")
    return pd.DataFrame(data)


def fetch_object_coords(dia_object_id):
    """
    Récupère RA, Dec et le filtre dominant pour un diaObjectId.
    On demande ra, dec, band et psfFlux via /api/v1/sources.
    """
    payload = {
        "diaObjectId"  : str(dia_object_id),
        "columns"      : "r:ra,r:dec,r:band,r:psfFlux,r:midpointMjdTai",
        "output-format": "json",
    }
    resp = requests.post(f"{BASE_URL}/api/v1/sources", json=payload, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    if not data:
        return None
    df = pd.DataFrame(data)
    df.columns = [c.replace('r:', '') for c in df.columns]
    for col in ['ra', 'dec', 'psfFlux', 'midpointMjdTai']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    # Coordonnées médianes (robuste aux outliers)
    ra_med  = df['ra'].median()
    dec_med = df['dec'].median()
    # Filtre dominant = filtre avec le plus de détections
    dom_filter = df['band'].mode()[0] if 'band' in df.columns and not df['band'].isna().all() else 'N/A'
    # Flux max
    flux_max = df['psfFlux'].max() if 'psfFlux' in df.columns else np.nan
    # Date de la dernière alerte
    last_mjd = df['midpointMjdTai'].max() if 'midpointMjdTai' in df.columns else np.nan
    return {
        'diaObjectId'  : dia_object_id,
        'ra'           : ra_med,
        'dec'          : dec_med,
        'filter_dom'   : dom_filter,
        'flux_max'     : flux_max,
        'last_mjd'     : last_mjd,
        'n_sources'    : len(df),
    }


from tqdm.notebook import tqdm

# ── Récupération ──────────────────────────────────────────────────────────────
print(f"Récupération de {N_ALERTS} alertes '{TAG}'...")
df_tags = fetch_tag_alerts(TAG, N_ALERTS, STARTDATE, STOPDATE)
id_col  = [c for c in df_tags.columns if 'diaObjectId' in c][0]
object_ids = df_tags[id_col].astype(str).unique().tolist()
print(f"✓ {len(object_ids)} objet(s) unique(s) trouvé(s)\n")

records = []
for oid in tqdm(object_ids, desc='Récupération coordonnées', unit='obj'):
    try:
        rec = fetch_object_coords(oid)
        if rec and not np.isnan(rec['ra']) and not np.isnan(rec['dec']):
            records.append(rec)
            #print(f"  ✓  {oid}  RA={rec['ra']:.4f}  Dec={rec['dec']:+.4f}  "
            #      f"filtre={rec['filter_dom']}  flux_max={rec['flux_max']:.1f} nJy")
        else:
            print(f"  ⚠️  {oid} — coordonnées manquantes")
    except Exception as e:
        print(f"  ✗  {oid} — erreur : {e}")

df_coords = pd.DataFrame(records)
print(f"\n{len(df_coords)} objet(s) avec coordonnées valides.")
ipy_display(df_coords.head())

## 4. Construction des courbes de superposition

Plan galactique et écliptique calculés via **astropy** et convertis en RA/Dec équatorial.

In [ ]:
def galactic_plane_radec(n=1000):
    """
    Retourne (ra_deg, dec_deg) du plan galactique (b=0)
    en coordonnées équatoriales ICRS.
    """
    l = np.linspace(0, 360, n)
    b = np.zeros(n)
    gc = SkyCoord(l=l*u.deg, b=b*u.deg, frame='galactic')
    eq = gc.icrs
    # Trier par RA pour un tracé continu
    idx = np.argsort(eq.ra.deg)
    return eq.ra.deg[idx], eq.dec.deg[idx]


def ecliptic_radec(n=1000):
    """
    Retourne (ra_deg, dec_deg) de l'écliptique (lat=0)
    en coordonnées équatoriales ICRS.
    """
    lon = np.linspace(0, 360, n)
    lat = np.zeros(n)
    ecl = SkyCoord(lon=lon*u.deg, lat=lat*u.deg,
                   frame=BarycentricTrueEcliptic(equinox='J2000'))
    eq  = ecl.icrs
    idx = np.argsort(eq.ra.deg)
    return eq.ra.deg[idx], eq.dec.deg[idx]


def radec_to_mollweide(ra_deg, dec_deg):
    """
    Convertit RA (0-360°) et Dec (-90/+90°) en coordonnées Mollweide.
    Convention astronomique : RA croissant vers la gauche (x inversé).
    Retourne (x_rad, y_rad) prêts pour ax.plot() sur un axe Mollweide.
    """
    ra  = np.asarray(ra_deg,  dtype=float)
    dec = np.asarray(dec_deg, dtype=float)
    # Centrage sur RA=180° et inversion (convention)
    x = -np.deg2rad(ra - 180.0)
    y =  np.deg2rad(dec)
    return x, y


# Pré-calcul des courbes
gal_ra,  gal_dec  = galactic_plane_radec()
ecl_ra,  ecl_dec  = ecliptic_radec()

print("Courbes de superposition calculées ✓")

## 5. Carte de Mollweide

In [ ]:
if df_coords.empty:
    print("⚠️  Aucune coordonnée disponible — relancer la section 3.")
else:
    fig = plt.figure(figsize=(16, 8), layout='constrained')
    ax  = fig.add_subplot(111, projection='mollweide')

    fig.suptitle(
        f"Localisation céleste — {TAG}\n"
        f"{len(df_coords)} objet(s) | Source : Fink/LSST portal"
    )

    # ── Grille ──────────────────────────────────────────────────────────────
    if SHOW_GRID:
        ax.grid(True, color='gray', alpha=0.35, linewidth=0.6, linestyle='--')

    # ── Plan galactique ──────────────────────────────────────────────────────
    if SHOW_GALACTIC_PLANE:
        gx, gy = radec_to_mollweide(gal_ra, gal_dec)
        # Le plan galactique croise RA=0 → on coupe pour éviter les traits parasites
        breaks = np.where(np.abs(np.diff(gx)) > np.pi * 0.8)[0] + 1
        segs_x = np.split(gx, breaks)
        segs_y = np.split(gy, breaks)
        for sx, sy in zip(segs_x, segs_y):
            ax.plot(sx, sy, color='goldenrod', lw=1.4,
                    label='Plan galactique' if sx is segs_x[0] else '')

    # ── Écliptique ───────────────────────────────────────────────────────────
    if SHOW_ECLIPTIC:
        ex, ey = radec_to_mollweide(ecl_ra, ecl_dec)
        breaks = np.where(np.abs(np.diff(ex)) > np.pi * 0.8)[0] + 1
        segs_x = np.split(ex, breaks)
        segs_y = np.split(ey, breaks)
        for sx, sy in zip(segs_x, segs_y):
            ax.plot(sx, sy, color='tomato', lw=1.0, ls='--',
                    label='Écliptique' if sx is segs_x[0] else '')

    # ── Alertes ──────────────────────────────────────────────────────────────
    if COLOR_BY == 'filter':
        # Couleur par filtre dominant
        for _, row in df_coords.iterrows():
            px, py = radec_to_mollweide(row['ra'], row['dec'])
            color  = FILTER_COLORS.get(row['filter_dom'], '#888888')
            # Croix noire = position de la galaxie hôte
            # Cercle coloré = alerte
            ax.plot(px, py, 'o', color=color, ms=6, alpha=0.85,
                    zorder=5, mec='white', mew=0.4)

        # Légende filtres
        used_filters = df_coords['filter_dom'].dropna().unique()
        handles = [
            plt.Line2D([0],[0], marker='o', color='w',
                       markerfacecolor=FILTER_COLORS.get(f,'#888'),
                       markersize=8, label=f"filtre {f}")
            for f in sorted(used_filters)
        ]

    elif COLOR_BY == 'flux':
        # Couleur par flux max (colormap continue)
        fluxes = df_coords['flux_max'].fillna(0)
        norm   = mcolors.LogNorm(vmin=max(fluxes.min(), 1), vmax=fluxes.max())
        cmap   = cm.plasma
        for _, row in df_coords.iterrows():
            px, py = radec_to_mollweide(row['ra'], row['dec'])
            color  = cmap(norm(max(row['flux_max'], 1)))
            ax.plot(px, py, 'o', color=color, ms=6, alpha=0.85,
                    zorder=5, mec='white', mew=0.4)
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        plt.colorbar(sm, ax=ax, orientation='horizontal',
                     pad=0.05, fraction=0.03, label='Flux max PSF (nJy)')
        handles = []

    # Entrée de légende pour la croix galaxie
    if SHOW_GALACTIC_PLANE:
        handles.append(plt.Line2D([0],[0], color='goldenrod', lw=1.4,
                                   label='Plan galactique'))
    if SHOW_ECLIPTIC:
        handles.append(plt.Line2D([0],[0], color='tomato', lw=1.0, ls='--',
                                   label='Écliptique'))
    ax.legend(handles=handles, loc='lower left', fontsize=9,
              framealpha=0.7, ncol=2)

    # ── Annotations α/δ ──────────────────────────────────────────────────────
    if SHOW_RADEC_LABELS or SHOW_OBJECT_ID:
        for _, row in df_coords.iterrows():
            px, py = radec_to_mollweide(row['ra'], row['dec'])
            parts  = []
            if SHOW_OBJECT_ID:
                parts.append(str(row['diaObjectId']))
            if SHOW_RADEC_LABELS:
                parts.append(f"α={row['ra']:.2f}°  δ={row['dec']:+.2f}°")
            ax.annotate(
                '\n'.join(parts),
                xy=(px, py), xytext=(4, 4), textcoords='offset points',
                fontsize=7, color='#222222', alpha=0.85,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.5, lw=0)
            )


    if SAVE_FIGURE:
        stem = f"{OUTPUT_DIR}/skymap_{TAG.replace(' ','_')}"
        fig.savefig(f"{stem}.pdf", bbox_inches='tight', dpi=150)
        fig.savefig(f"{stem}.png", bbox_inches='tight', dpi=150)
        print(f"✓ Sauvegardé : {stem}.pdf / .png")

    plt.show()

## 6. Tableau récapitulatif

In [ ]:
if not df_coords.empty:
    MJD_EPOCH = pd.Timestamp('1858-11-17', tz='UTC')
    df_display = df_coords.copy()
    df_display['last_date'] = (
        MJD_EPOCH + pd.to_timedelta(df_display['last_mjd'], unit='D')
    ).dt.strftime('%Y-%m-%d')

    df_show = df_display[[
        'diaObjectId', 'ra', 'dec', 'filter_dom',
        'flux_max', 'n_sources', 'last_date'
    ]].copy()
    df_show['ra']       = df_show['ra'].round(5)
    df_show['dec']      = df_show['dec'].round(5)
    df_show['flux_max'] = df_show['flux_max'].round(1)

    # Lien Fink portal (HTML manuel — pas besoin de jinja2)
    df_show['Portail Fink'] = df_show['diaObjectId'].apply(
        lambda oid: f'<a href="https://lsst.fink-portal.org/{oid}" target="_blank">'
                    f'lsst.fink-portal.org/{oid}</a>'
    )

    df_show = df_show.rename(columns={
        'ra'         : 'RA (deg)',
        'dec'        : 'Dec (deg)',
        'filter_dom' : 'Filtre dom.',
        'flux_max'   : 'Flux max (nJy)',
        'n_sources'  : 'N sources',
        'last_date'  : 'Dernière alerte',
    })

    from IPython.display import HTML
    html = df_show.to_html(index=False, escape=False,
                           border=0, classes='table')
    # Style CSS minimal pour lisibilité
    style = """
    <style>
      .table { border-collapse: collapse; font-size: 12px; width: 100%; }
      .table th { background: #f0f0f0; padding: 6px 10px;
                  border-bottom: 2px solid #ccc; text-align: left; }
      .table td { padding: 4px 10px; border-bottom: 1px solid #eee;
                  text-align: left; }
      .table tr:hover td { background: #fafafa; }
    </style>
    """
    ipy_display(HTML(style + html))


## 7. Distributions angulaires (RA et Dec)

In [ ]:
if not df_coords.empty:
    fig, (ax_ra, ax_dec) = plt.subplots(1, 2, figsize=(13, 4), layout='constrained')
    fig.suptitle(f"Distributions angulaires — {TAG}")

    # ── RA ───────────────────────────────────────────────────────────────────
    ax_ra.hist(df_coords['ra'], bins=20, color='steelblue', edgecolor='white',
               linewidth=0.5, alpha=0.85)
    ax_ra.set_xlabel("Ascension droite α (deg)")
    ax_ra.set_ylabel("N objets")
    ax_ra.set_xlim(0, 360)
    ax_ra.grid(True, alpha=0.25, linewidth=0.5)

    # ── Dec ──────────────────────────────────────────────────────────────────
    ax_dec.hist(df_coords['dec'], bins=20, color='darkorange', edgecolor='white',
                linewidth=0.5, alpha=0.85)
    ax_dec.set_xlabel("Déclinaison δ (deg)")
    ax_dec.set_ylabel("N objets")
    ax_dec.set_xlim(-90, 90)
    ax_dec.axvline(-62, color='goldenrod', lw=1.2, ls='--',
                   label='Limite LSST (~-62°)')  # limite indicative champ LSST
    ax_dec.legend(fontsize=9)
    ax_dec.grid(True, alpha=0.25, linewidth=0.5)

    if SAVE_FIGURE:
        stem = f"{OUTPUT_DIR}/histo_radec_{TAG.replace(' ','_')}"
        fig.savefig(f"{stem}.pdf", bbox_inches='tight', dpi=150)
        fig.savefig(f"{stem}.png", bbox_inches='tight', dpi=150)
        print(f"✓ Sauvegardé : {stem}.pdf / .png")

    plt.show()